### Import Required Dependencies

In [1]:
# --- Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import re


from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# --- Logging ---
logging.basicConfig(
    filename=log_dir / "project.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode='w'
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("[%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)

print("✅ Environment ready. Paths and logging configured.")



✅ Environment ready. Paths and logging configured.


# Section 2 Load and Process Dataset (national and Arkansas)

## Section 2A: Load utility functions

In [2]:
# Section 2A: Load utility functions and national datasets ----

# Import custom utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,
    filter_to_county_level,
    log_duplicate_attributes,
    clean_and_extract_year,
    nca_counties
)

data_dir = Path("data")
complete_dir = data_dir / "complete_sets"

complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

# Load, clean, and extract year from each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        logging.info(f"📌 {key} columns after cleaning: {df.columns.tolist()}")
        df = filter_to_county_level(df)
        df = clean_and_extract_year(df)

        # Rename fields if necessary
        df.rename(columns={'area_name': 'county'}, inplace=True)

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"✅ Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")


# 4. Confirmation message
logging.info("🧠 Utility functions from utils.py loaded successfully.")


[INFO] ✅ Utility functions loaded from updated utils.py
[INFO] 📌 edu columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
[WARNING] ⚠️ edu has duplicate county-attribute pairs.
[INFO] ✅ Loaded Education2023.csv: 169245 rows
[INFO] 📌 pop columns after cleaning: ['fipstxt', 'state', 'area_name', 'attribute', 'value']
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[INFO] ✅ Loaded PopulationEstimates.csv: 205108 rows
[INFO] 📌 poverty columns after cleaning: ['ïfips_code', 'stabr', 'area_name', 'attribute', 'value']
[WARNING] ⚠️ poverty has duplicate county-attribute pairs.
[INFO] ✅ Loaded Poverty2023.csv: 79961 rows
[INFO] 📌 unemp columns after cleaning: ['fips_code', 'state', 'area_name', 'attribute', 'value']
[INFO] ✅ Loaded Unemployment2023.csv: 324636 rows
[INFO] 🧠 Utility functions from utils.py loaded successfully.


## Section 2B: Load and process national datasets

In [3]:
# Section 2B: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

complete_data = {}
state_lookup = None

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'fips_code': 'fips', 'fipstxt': 'fips', 'area_name': 'county'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()
        df['attribute'] = df['attribute'].str.strip().str.lower()

        # 🔎 Only keep rows for 2022
        df = df[df['attribute'].str.contains("2022", na=False)]

        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.strip().str.lower()
            state_lookup['state'] = state_lookup['state'].str.strip().str.upper()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"✅ Loaded and filtered 2022 rows from {filename}")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")

# Pivot and merge only 2022 data
edu_wide = complete_data['edu'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
pop_wide = complete_data['pop'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
poverty_wide = complete_data['poverty'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')
unemp_wide = complete_data['unemp'].pivot_table(index='county', columns='attribute', values='value', aggfunc='first')

df_full = reduce(lambda l, r: pd.merge(l, r, on='county', how='outer'),
                 [edu_wide, pop_wide, poverty_wide, unemp_wide])

df_full = df_full.reset_index().merge(state_lookup, on='county', how='left')
df_full['state'] = df_full['state'].str.upper()
df_full.set_index('county', inplace=True)

logging.info(f"✅ Final dataset shape (2022 only): {df_full.shape}")
df_full.to_csv(output_dir / "us_county_merged.csv")



[INFO] ✅ Loaded and filtered 2022 rows from Education2023.csv
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[INFO] ✅ Loaded and filtered 2022 rows from PopulationEstimates.csv
[INFO] ✅ Loaded and filtered 2022 rows from Poverty2023.csv
[INFO] ✅ Loaded and filtered 2022 rows from Unemployment2023.csv
[INFO] ✅ Final dataset shape (2022 only): (5189, 23)


### Section 2B.1 Determine most common year across all datasets

In [4]:
## 📊 Section 2B.1: Discover Common Year Across Datasets

import re
from collections import Counter

def extract_years_from_attributes(df):
    year_pattern = re.compile(r'(?:19|20)\d{2}')
    years = set()
    if 'attribute' in df.columns:
        for val in df['attribute'].dropna():
            found = year_pattern.findall(str(val))
            years.update([y for y in found if int(y) >= 2000])
    return years

year_sets = {}
print("📊 Year sets by dataset:")

for key, df in complete_data.items():
    years = extract_years_from_attributes(df)
    year_sets[key] = years
    print(f"{key.upper()}: {sorted(years) if years else '❌ No year tags'}")

# ✅ Determine which year(s) are shared
datasets_with_years = [v for v in year_sets.values() if v]
if datasets_with_years:
    common_years = set.intersection(*datasets_with_years)
    print("\n✅ Common years across all datasets that include year tags:", sorted(common_years))
else:
    print("\n⚠️ None of the datasets have year-tagged attributes.")



📊 Year sets by dataset:
EDU: ❌ No year tags
POP: ['2022']
POVERTY: ❌ No year tags
UNEMP: ['2022']

✅ Common years across all datasets that include year tags: ['2022']


## Section 2C: Subset Arkansas and NCA Counties

In [5]:
# Section 2C: Subset Arkansas and NCA counties

df_ar = df_full[df_full['state'] == 'AR'].copy()
df_ar = df_ar.reset_index()
df_ar['county'] = df_ar['county'].str.replace(" county", "", regex=False).str.replace(", ar", "", regex=False).str.strip()
df_ar.set_index('county', inplace=True)

df_nca = df_ar[df_ar.index.isin(nca_counties)].copy()
df_ar.to_csv(output_dir / "arkansas_counties.csv")
df_nca.to_csv(output_dir / "nca_counties.csv")

logging.info(f"📌 Arkansas counties: {df_ar.shape[0]}")
logging.info(f"📌 NCA counties: {df_nca.shape[0]}")


[INFO] 📌 Arkansas counties: 0
[INFO] 📌 NCA counties: 0


## Section 3: Exploratory Data Analysis (EDA)

In [6]:
# SECTION 3: Exploratory Data Analysis (EDA)

# Use a working copy of the NCA subset
df = df_nca.copy()
logging.info(f"🔍 Starting EDA on NCA dataset: {df.shape[0]} counties, {df.shape[1]} features")


[INFO] 🔍 Starting EDA on NCA dataset: 0 counties, 23 features


### Section 3.1: Inspect Dataset

In [7]:
# 🧠 Full dataset overview
print("🔧 Info:")
display(df.info())

print("\n📊 Descriptive Stats:")
display(df.describe(include='all'))

print("\n🔍 Sample Data:")
display(df.head())


🔧 Info:
<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 23 columns):
 #   Column                                     Non-Null Count  Dtype  
---  ------                                     --------------  -----  
 0   births_2022                                0 non-null      float64
 1   deaths_2022                                0 non-null      float64
 2   domestic_mig_2022                          0 non-null      float64
 3   gq_estimates_2022                          0 non-null      float64
 4   international_mig_2022                     0 non-null      float64
 5   n_pop_chg_2022                             0 non-null      float64
 6   natural_chg_2022                           0 non-null      float64
 7   net_mig_2022                               0 non-null      float64
 8   pop_estimate_2022                          0 non-null      float64
 9   r_birth_2022                               0 non-null      float64
 10  r_death_2022                       

None


📊 Descriptive Stats:


,births_2022,deaths_2022,domestic_mig_2022,gq_estimates_2022,international_mig_2022,n_pop_chg_2022,natural_chg_2022,net_mig_2022,pop_estimate_2022,r_birth_2022,...,r_natural_chg_2022,r_net_mig_2022,residual_2022,civilian_labor_force_2022,employed_2022,med_hh_income_percent_of_state_total_2022,median_household_income_2022,unemployed_2022,unemployment_rate_2022,state
count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



🔍 Sample Data:


,births_2022,deaths_2022,domestic_mig_2022,gq_estimates_2022,international_mig_2022,n_pop_chg_2022,natural_chg_2022,net_mig_2022,pop_estimate_2022,r_birth_2022,...,r_natural_chg_2022,r_net_mig_2022,residual_2022,civilian_labor_force_2022,employed_2022,med_hh_income_percent_of_state_total_2022,median_household_income_2022,unemployed_2022,unemployment_rate_2022,state
county,,,,,,,,,,,,,,,,,,,,,


### Section 3.2 Extract and Rename Key Variables

In [8]:
## Section 3.2: Extract and Name Key Indicator Variables

# 📂 Print all column names for inspection
print("\n📂 All column names in df_nca:")
for col in df_nca.columns:
    print(col)

# 🔍 Extract key indicators
df_nca, used_columns, year = extract_key_indicators(df_nca, min_year=2022)

# 📊 Print matched column summary
print("📅 Most common year in column names:", year)
print("📚 Education columns used:", used_columns['education'])
print("📉 Poverty columns used:", used_columns['poverty'])
print("💼 Unemployment columns used:", used_columns['unemployment'])
print("👥 Population columns used:", used_columns['population'])

# 🧮 Compute education percentages (relative to population)
df_nca['BachelorsDegreePct'] = (df_nca['BachelorsDegreeRate'] / df_nca['Population']) * 100
df_nca['HighSchoolGradPct'] = (df_nca['HighSchoolGradRate'] / df_nca['Population']) * 100

# ✅ Define clean variable set for EDA (use percentages)
variables = ['BachelorsDegreePct', 'HighSchoolGradPct', 'PovertyRate', 'UnemploymentRate', 'Population']

# 🔎 Preview the cleaned dataset
print("\n🔎 Preview of standardized indicators:")
display(df_nca[variables].head())

# 🧭 Debug: Print all matched and available columns
print("\n📚 Matched columns by indicator:")
for key, cols in used_columns.items():
    print(f"  - {key}: {cols}")

print("\n📁 All available columns:")
print(df_nca.columns.tolist())



[INFO] 📅 Most common year in column names: None
[INFO] 📚 Education columns: []
[INFO] 📉 Poverty columns: []
[INFO] 💼 Unemployment columns: []
[INFO] 👥 Population columns: []
[WARNING] ⚠️ Unemployment column not matched by regex. Using fallback 'unemployment_rate_2022'.



📂 All column names in df_nca:
births_2022
deaths_2022
domestic_mig_2022
gq_estimates_2022
international_mig_2022
n_pop_chg_2022
natural_chg_2022
net_mig_2022
pop_estimate_2022
r_birth_2022
r_death_2022
r_domestic_mig_2022
r_international_mig_2022
r_natural_chg_2022
r_net_mig_2022
residual_2022
civilian_labor_force_2022
employed_2022
med_hh_income_percent_of_state_total_2022
median_household_income_2022
unemployed_2022
unemployment_rate_2022
state
📅 Most common year in column names: None
📚 Education columns used: []
📉 Poverty columns used: []
💼 Unemployment columns used: []
👥 Population columns used: []

🔎 Preview of standardized indicators:


,BachelorsDegreePct,HighSchoolGradPct,PovertyRate,UnemploymentRate,Population
county,,,,,



📚 Matched columns by indicator:
  - education: []
  - poverty: []
  - unemployment: []
  - population: []

📁 All available columns:
['births_2022', 'deaths_2022', 'domestic_mig_2022', 'gq_estimates_2022', 'international_mig_2022', 'n_pop_chg_2022', 'natural_chg_2022', 'net_mig_2022', 'pop_estimate_2022', 'r_birth_2022', 'r_death_2022', 'r_domestic_mig_2022', 'r_international_mig_2022', 'r_natural_chg_2022', 'r_net_mig_2022', 'residual_2022', 'civilian_labor_force_2022', 'employed_2022', 'med_hh_income_percent_of_state_total_2022', 'median_household_income_2022', 'unemployed_2022', 'unemployment_rate_2022', 'state', 'BachelorsDegreeRate', 'HighSchoolGradRate', 'PovertyRate', 'UnemploymentRate', 'Population', 'Year', 'BachelorsDegreePct', 'HighSchoolGradPct']


### Section 3.3 Visualize Education Levels

In [9]:
# Education distribution across counties
title = "Education Indicators by County"
df[education_cols].T.plot(kind='bar', figsize=(14, 6), title=title)
plt.ylabel("Percent or Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Standardize filename
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f" Saved: {filename}.png")

plt.show()



NameError: name 'education_cols' is not defined

In [ ]:
print('unemployment_rate_2023' in df_nca.columns)  # Should return True
print(df_nca['unemployment_rate_2023'].head())     # Preview values


### Section 3.4: Distribution Plots (Histograms & KDE)

In [ ]:
# Distribution plots with title-based saving
variables = ['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate']

for var in variables:
    plt.figure(figsize=(8, 4))
    
    # Define plot title
    title = f"Distribution of {var}"
    
    # Plot
    sns.histplot(df[var], kde=True, bins=20)
    plt.title(title)
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.tight_layout()
    
    # Create safe filename from title
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")
    
    plt.show()


### Section 3.5: Correlation Heatmap

In [ ]:
# Correlation matrix and heatmap
corr_vars = df[['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate', 'Population']]
corr_matrix = corr_vars.corr()

# Define title
title = "Correlation Between Key Indicators"

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title(title)
plt.tight_layout()

# Generate safe filename from title
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")

plt.show()


### Section 3.6: Key Relationships (Scatter Plots)

In [ ]:
# Scatter Plots of Key Relationships
# 1. Bachelor's Degree vs Poverty
title = "Bachelor's Degree Rate vs. Poverty Rate"
sns.scatterplot(x='BachelorsDegreeRate', y='PovertyRate', data=df)
plt.title(title)
plt.xlabel("Bachelor's Degree (%)")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 2. High School Grad Rate vs Unemployment
title = "High School Grad Rate vs. Unemployment Rate"
sns.scatterplot(x='HighSchoolGradRate', y='UnemploymentRate', data=df)
plt.title(title)
plt.xlabel("High School Grad (%)")
plt.ylabel("Unemployment Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 3. Population vs Poverty Rate
title = "Population vs. Poverty Rate"
sns.scatterplot(x='Population', y='PovertyRate', data=df)
plt.title(title)
plt.xlabel("Population")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()


### Section 3.7: Outlier Detection with BoxPlots

In [ ]:
# --- Boxplots for Outlier Detection (with title-based save) ---
for var in variables:
    plt.figure(figsize=(8, 4))

    # Define title and filename
    title = f"Boxplot of {var}"
    sns.boxplot(x=df[var])
    plt.title(title)
    plt.tight_layout()

    # Standardize filename
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")

    plt.show()


### Section 3.8 ParPlot for Key Indicators

In [ ]:
# Pairplot for Key Indicators
sns.pairplot(df[variables], diag_kind='kde')
plt.suptitle("Pairwise Relationships Between Key Indicators", y=1.02)
plt.tight_layout()
plt.savefig(image_dir / "pairplot_key_indicators.png", dpi=300, bbox_inches='tight')
logging.info("📷 Saved: pairplot_key_indicators.png")
plt.show()


### Step 4: Visualize Distributions (Histograms & KDE)

### Step 5: Correlation Matrix and Heatmap

### Step 6: Scatter Plots for Key Relationships

### Step 7: Identify Outlier Counties with Boxplots

### Step 8: Log-Transform Population (Optional)
If the population is skewed: